# Day 10: SVM & KNN

## Task 1: Iris Dataset - SVM & KNN Comparison

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
iris = load_iris()

X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### Part A: SVM Hyperparameter Tuning

In [ ]:
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1]
}

svm = SVC(kernel='rbf')

grid = GridSearchCV(
    svm,
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)

print("Best CV Score:")
print(grid.best_score_)

best_svm = grid.best_estimator_

svm_test_accuracy = best_svm.score(X_test, y_test)

print("Test Accuracy:")
print(svm_test_accuracy)

### Part B: Optimal K for KNN

In [ ]:
k_values = range(1, 21)

accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    accuracy = knn.score(X_test, y_test)
    accuracies.append(accuracy)

best_k = k_values[np.argmax(accuracies)]

print("Best K =", best_k)
print("Best Accuracy =", max(accuracies))

plt.figure(figsize=(8,5))
plt.plot(k_values, accuracies, marker='o')
plt.xlabel("K Value")
plt.ylabel("Accuracy")
plt.title("K vs Accuracy")
plt.grid(True)
plt.show()

### Part C: 5-Fold Cross Validation Comparison

In [ ]:
svm_model = SVC(
    kernel='rbf',
    C=grid.best_params_['C'],
    gamma=grid.best_params_['gamma']
)

knn_model = KNeighborsClassifier(n_neighbors=best_k)

lr_model = LogisticRegression(max_iter=1000)

svm_cv = cross_val_score(svm_model, X, y, cv=5, scoring='accuracy')
knn_cv = cross_val_score(knn_model, X, y, cv=5, scoring='accuracy')
lr_cv = cross_val_score(lr_model, X, y, cv=5, scoring='accuracy')

print("SVM Average Accuracy:")
print(svm_cv.mean())

print("KNN Average Accuracy:")
print(knn_cv.mean())

print("Logistic Regression Average Accuracy:")
print(lr_cv.mean())

### Part D: Manhattan vs Euclidean Distance

In [ ]:
knn_euclidean = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean')

knn_manhattan = KNeighborsClassifier(n_neighbors=best_k, metric='manhattan')

knn_euclidean.fit(X_train, y_train)
knn_manhattan.fit(X_train, y_train)

euclidean_acc = knn_euclidean.score(X_test, y_test)

manhattan_acc = knn_manhattan.score(X_test, y_test)

print("Euclidean Accuracy:")
print(euclidean_acc)

print("Manhattan Accuracy:")
print(manhattan_acc)

if manhattan_acc > euclidean_acc:
    print("Manhattan performs better")
elif euclidean_acc > manhattan_acc:
    print("Euclidean performs better")
else:
    print("Both perform equally")

## Task 2: Curse of Dimensionality & PCA

In [ ]:
import numpy as np

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA

In [ ]:
X, y = make_classification(
    n_samples=500,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

### Accuracy with 2 Features (Original)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

original_accuracy = accuracy_score(y_test, y_pred)

print("Accuracy with 2 Features:")
print(original_accuracy)

### Adding 50 Noise Features (Curse of Dimensionality)

In [ ]:
X_noisy = np.hstack([
    X,
    np.random.randn(500, 50)
])

Xn_train, Xn_test, yn_train, yn_test = train_test_split(
    X_noisy,
    y,
    test_size=0.2,
    random_state=42
)

knn_noisy = KNeighborsClassifier(n_neighbors=5)

knn_noisy.fit(Xn_train, yn_train)

yn_pred = knn_noisy.predict(Xn_test)

noisy_accuracy = accuracy_score(yn_test, yn_pred)

print("Accuracy with 52 Features:")
print(noisy_accuracy)

print("Accuracy Drop:")
print(original_accuracy - noisy_accuracy)

### Recovering Accuracy with PCA

In [ ]:
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_noisy)

Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    X_pca,
    y,
    test_size=0.2,
    random_state=42
)

knn_pca = KNeighborsClassifier(n_neighbors=5)

knn_pca.fit(Xp_train, yp_train)

yp_pred = knn_pca.predict(Xp_test)

pca_accuracy = accuracy_score(yp_test, yp_pred)

print("Accuracy after PCA:")
print(pca_accuracy)

print("Recovery:")
print(pca_accuracy - noisy_accuracy)